# Đối soát số liệu paper (không huấn luyện)

Chạy lần lượt các ô. Notebook chỉ đọc artifact hiện có, không tải model và không sửa file kết quả. Các run khác nhau được giữ riêng. Chỉ ghép prediction theo ảnh khi chắc chắn cùng thứ tự test ID.

Đầu ra là phép tính độc lập từ JSON và tensor; không thay thế việc xác minh nguồn gốc dữ liệu/cache.

In [1]:
from pathlib import Path
import json
import torch

def find_repo(start=Path.cwd()):
    for p in (start, *start.parents):
        if (p / "paper/paper_CLIP_Gemma4_revised_v2.md").exists() and (p / "Semester/results").exists():
            return p
    raise FileNotFoundError("Mở notebook từ trong repository CLIP-Animal-Classifier")

ROOT = find_repo()
RESULTS = ROOT / "Semester/results"
def load_json(name):
    return json.loads((RESULTS / name).read_text(encoding="utf-8"))
print("Repository:", ROOT.name)
print("Paper:", (ROOT / "paper/paper_CLIP_Gemma4_revised_v2.md").exists())

Repository: CLIP-Animal-Classifier
Paper: True


In [2]:
table7 = load_json("table7_full_benchmark_results.json")
models = table7["models"]
budgets = ["1_shot", "2_shot", "4_shot", "8_shot", "16_shot", "full_trainval"]
print("Table 7: mean theo JSON (điểm phần trăm)")
print("Budget | Gemma-CLIP Full | CoOp | Delta")
for k in budgets:
    gemma = models["Gemma-CLIP Full"][k]["mean"]
    coop = models["CLIP + CoOp"][k]["mean"]
    print(f"{k:13s} | {gemma:6.2f} | {coop:6.2f} | {gemma-coop:+6.2f}")
print("K=0 trong JSON:", table7.get("clip_handcrafted_zero_shot"), "(không mặc định là cùng protocol với K=0 của paper)")

Table 7: mean theo JSON (điểm phần trăm)
Budget | Gemma-CLIP Full | CoOp | Delta
1_shot        |  85.83 |  86.61 |  -0.78
2_shot        |  87.55 |  88.51 |  -0.96
4_shot        |  88.66 |  89.78 |  -1.12
8_shot        |  88.93 |  90.18 |  -1.25
16_shot       |  89.52 |  90.87 |  -1.35
full_trainval |  90.07 |  91.25 |  -1.18
K=0 trong JSON: 87.49 (không mặc định là cùng protocol với K=0 của paper)


In [3]:
def pred_audit(filename, pairs):
    d = torch.load(RESULTS / filename, map_location="cpu", weights_only=True)
    y = d["test_labels"]
    print("\n", filename, "N=", len(y))
    for name in [k for k in d if k.startswith("pred_")]:
        correct = int((d[name] == y).sum())
        print(f"  {name}: {correct}/{len(y)} = {100*correct/len(y):.4f}%")
    for a,b in pairs:
        ac, bc = d[a] == y, d[b] == y
        n10 = int((ac & ~bc).sum())
        n01 = int((~ac & bc).sum())
        chi2 = (abs(n10-n01)-1)**2/(n10+n01) if n10+n01 else 0
        print(f"  {a} vs {b}: b={n10}, c={n01}, b-c={n10-n01}, chi2={chi2:.4f}")
    return d

old = pred_audit("test_predictions_verified.pt", [("pred_dual","pred_coop_base"), ("pred_dual","pred_res")])
new = pred_audit("verified_test_predictions.pt", [("pred_dual","pred_coop"), ("pred_dual","pred_res")])
print("\nLưu ý: hai file có ResNet prediction khác nhau; không trộn b/c hoặc accuracy giữa chúng.")


 test_predictions_verified.pt N= 3669
  pred_coop_base: 3353/3669 = 91.3873%
  pred_gcoop: 3365/3669 = 91.7144%
  pred_dual: 3365/3669 = 91.7144%
  pred_res: 3255/3669 = 88.7163%
  pred_dual vs pred_coop_base: b=100, c=88, b-c=12, chi2=0.6436
  pred_dual vs pred_res: b=270, c=160, b-c=110, chi2=27.6302

 verified_test_predictions.pt N= 3669
  pred_dual: 3365/3669 = 91.7144%
  pred_coop: 3353/3669 = 91.3873%
  pred_res: 3174/3669 = 86.5086%
  pred_dual vs pred_coop: b=100, c=88, b-c=12, chi2=0.6436
  pred_dual vs pred_res: b=352, c=161, b-c=191, chi2=70.3704

Lưu ý: hai file có ResNet prediction khác nhau; không trộn b/c hoặc accuracy giữa chúng.


In [4]:
classes = json.loads((ROOT / "Semester/cached_features/oxford_pets/classes.json").read_text(encoding="utf-8"))
cat_names = {"abyssinian","bengal","birman","bombay","british_shorthair","egyptian_mau","maine_coon","persian","ragdoll","russian_blue","siamese","sphynx"}
cat_ids = {i for i,c in enumerate(classes) if c.lower().replace(" ","_") in cat_names}
y, p = new["test_labels"], new["pred_dual"]
cross = [(i, classes[int(t)], classes[int(pred)]) for i,(t,pred) in enumerate(zip(y,p)) if (int(t) in cat_ids) != (int(pred) in cat_ids)]
beagle = classes.index("Beagle")
beagle_errors = [classes[int(pred)] for pred,t in zip(p,y) if int(t)==beagle and int(pred)!=beagle]
print("Cross-species errors in verified_test_predictions.pt:", cross)
print("Beagle errors in same run:", beagle_errors)
print("Không áp kết quả run này lên một run khác nếu paper có prediction riêng.")

Cross-species errors in verified_test_predictions.pt: [(2673, 'Ragdoll', 'Samoyed'), (3013, 'Scottish Terrier', 'Abyssinian')]
Beagle errors in same run: ['German Shorthaired']
Không áp kết quả run này lên một run khác nếu paper có prediction riêng.


In [5]:
a = load_json("table5_alpha_verification.json")
print("Alpha file named by paper: alpha | validation | raw test | TTA key?")
for k,v in a.items():
    print(k, f"{v['val_acc']:.2f}", f"{v['test_raw_acc']:.2f}", "test_tta_acc" in v)
print("Tie on validation:", a["0.2"]["val_acc"] == a["0.4"]["val_acc"])
dogs = load_json("verified_stanford_dogs_dualstream.json")["models"]
print("\nStanford Dogs: count-derived top-1")
for k,v in dogs.items():
    print(k, v["correct_test"], "/", v["total_test"], "=", round(100*v["correct_test"]/v["total_test"],4), "%")
baseline = load_json("verified_cupl_labo_baselines.json")
print("\nCuPL/LaBo JSON có count; tên phương pháp cần đối chiếu script tạo prompt/concept riêng.")

Alpha file named by paper: alpha | validation | raw test | TTA key?
0.0 83.24 86.45 False
0.1 91.89 89.34 False
0.2 92.43 89.42 False
0.4 92.43 89.48 False
0.6 92.16 89.23 False
Tie on validation: True

Stanford Dogs: count-derived top-1
coop_baseline 3553 / 6174 = 57.5478 %
gemma_coop 3579 / 6174 = 57.9689 %
gemma_dualstream 3681 / 6174 = 59.621 %

CuPL/LaBo JSON có count; tên phương pháp cần đối chiếu script tạo prompt/concept riêng.


## Cách dùng kết quả

- Accuracy của hai lần train độc lập có thể so sánh nếu dataset, test IDs, backbone và protocol được mô tả; không cần gộp vào một script.
- McNemar đòi hai vector dự đoán căn đúng cùng ảnh test. Nếu chưa có ID ảnh, hãy kiểm manifest cache trước khi ghép.
- Notebook này không xác nhận CuPL/LaBo nguyên bản: script hiện lấy descriptor Gemma cho cả hai baseline.